# Notebook 02 — Construção e Validação do Grafo MST

**Propósito:** Verificar que o grafo MST tem estrutura setorial correta antes de calcular qualquer curvatura.

**Critérios de saída (todos devem ser verdadeiros para avançar ao Sprint 3):**
- [ ] Grafo MST é conexo
- [ ] VALE3 e PETR4 são vizinhos próximos na MST (cluster commodities)
- [ ] ITUB4 e BBDC4 são vizinhos próximos na MST (cluster bancos)
- [ ] Distância intra-setor < distância inter-setor
- [ ] Todas as distribuições de Markov somam 1
- [ ] Tempo de build < 5 segundos por janela

In [ ]:
import sys
sys.path.insert(0, '..')

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
from pathlib import Path

from src.data.loader import DataLoader
from src.data.cleaner import DataCleaner
from src.data.universe import UniverseBuilder
from src.graph.builder import GraphBuilder

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})

# Mapa de setores para colorir o grafo
SECTOR_MAP = {
    'VALE3': 'Commodities', 'VALE5': 'Commodities',
    'PETR4': 'Commodities', 'PETR3': 'Commodities',
    'CSAN3': 'Commodities', 'SUZB3': 'Materiais',
    'ITUB4': 'Bancos', 'ITUB3': 'Bancos',
    'BBDC4': 'Bancos', 'BBDC3': 'Bancos',
    'BBAS3': 'Bancos', 'SANB11': 'Bancos',
    'ABEV3': 'Consumo', 'BRFS3': 'Consumo', 'JBSS3': 'Consumo',
    'WEGE3': 'Industrial', 'EMBR3': 'Industrial',
    'RENT3': 'Serviços', 'TOTS3': 'Serviços',
    'MGLU3': 'Varejo', 'LREN3': 'Varejo', 'VIIA3': 'Varejo',
    'ELET3': 'Utilidades', 'ELET6': 'Utilidades', 'CPFE3': 'Utilidades',
    'RDOR3': 'Saúde', 'HAPV3': 'Saúde',
}
SECTOR_COLORS = {
    'Commodities': '#e74c3c', 'Materiais': '#e67e22',
    'Bancos': '#3498db', 'Consumo': '#2ecc71',
    'Industrial': '#9b59b6', 'Serviços': '#1abc9c',
    'Varejo': '#f39c12', 'Utilidades': '#7f8c8d',
    'Saúde': '#27ae60', 'Outros': '#bdc3c7',
}

print('Imports OK ✓')

In [ ]:
# ─── 1. CARREGAR DADOS PROCESSADOS ───────────────────────────
PROCESSED = Path('../data/processed')

returns = pd.read_parquet(PROCESSED / 'returns_b3.parquet')
universe = pd.read_csv(PROCESSED / 'universe_b3.csv')['ticker'].tolist()

print(f'Retornos: {returns.shape}')
print(f'Universo: {len(universe)} ativos')

In [ ]:
# ─── 2. CONSTRUIR GRAFO EM UMA DATA DE TESTE ─────────────────
builder = GraphBuilder()
cleaner = DataCleaner()

test_date = pd.Timestamp('2023-06-30')
window = cleaner.get_returns_window(returns, test_date, window_days=60)

print(f'Janela: {window.index[0].date()} → {window.index[-1].date()} ({len(window)} dias)')

# Medir tempo de build
t0 = time.perf_counter()
G = builder.build(window, eligible_tickers=universe)
elapsed = time.perf_counter() - t0

print(f'\nGrafo construído em {elapsed:.2f}s')
print(f'Nós: {G.number_of_nodes()}')
print(f'Arestas MST: {sum(1 for _,_,d in G.edges(data=True) if d.get("in_mst"))}')
print(f'Arestas extras: {sum(1 for _,_,d in G.edges(data=True) if not d.get("in_mst", True))}')

assert elapsed < 5.0, f'Build muito lento: {elapsed:.1f}s (limite: 5s)'

In [ ]:
# ─── 3. VALIDAÇÃO DE SANIDADE ────────────────────────────────
sanity = builder.validate_graph_sanity(G)
print('Checks de sanidade:')
for check, result in sanity.items():
    status = '✅' if result else ('⚠️ N/A' if result is None else '❌')
    print(f'  {status} {check}: {result}')

In [ ]:
# ─── 4. VISUALIZAÇÃO DO GRAFO MST ────────────────────────────
# Apenas arestas MST para clareza visual
G_mst_only = nx.Graph()
G_mst_only.add_nodes_from(G.nodes(data=True))
for u, v, d in G.edges(data=True):
    if d.get('in_mst', False):
        G_mst_only.add_edge(u, v, **d)

# Cores por setor
node_colors = [SECTOR_COLORS.get(SECTOR_MAP.get(n, 'Outros'), '#bdc3c7')
               for n in G_mst_only.nodes()]

# Layout
pos = nx.kamada_kawai_layout(G_mst_only, weight='weight')

# Espessura das arestas proporcional à correlação
edge_weights = [G_mst_only[u][v].get('correlation', 0.5) for u, v in G_mst_only.edges()]
edge_widths  = [max(0.5, c * 3) for c in edge_weights]

fig, ax = plt.subplots(figsize=(18, 14))

nx.draw_networkx_edges(
    G_mst_only, pos, ax=ax,
    width=edge_widths,
    alpha=0.4, edge_color='gray'
)
nx.draw_networkx_nodes(
    G_mst_only, pos, ax=ax,
    node_color=node_colors,
    node_size=600, alpha=0.9
)
nx.draw_networkx_labels(
    G_mst_only, pos, ax=ax,
    font_size=7, font_weight='bold'
)

# Legenda
legend_patches = [
    mpatches.Patch(color=color, label=sector)
    for sector, color in SECTOR_COLORS.items()
    if any(SECTOR_MAP.get(n) == sector for n in G_mst_only.nodes())
]
ax.legend(handles=legend_patches, loc='upper left', fontsize=9)

ax.set_title(
    f'Minimum Spanning Tree do Mercado Brasileiro\n'
    f'{test_date.date()} | {G_mst_only.number_of_nodes()} ativos | Janela: 60 dias úteis',
    fontsize=13, pad=20
)
ax.axis('off')

plt.tight_layout()
plt.savefig('../reports/pre_relatorio/figures/04_mst_graph.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n⚠ VALIDAÇÃO VISUAL OBRIGATÓRIA:')
print('  - Ativos do mesmo setor (mesma cor) devem estar próximos?')
print('  - Há clusters visíveis de bancos, commodities, consumo?')
print('  - Se não, verificar implementação da distância.')

In [ ]:
# ─── 5. ANÁLISE DE DISTÂNCIAS INTRA vs INTER SETOR ──────────
edges_df = builder.get_adjacency_info(G)

def get_sector(ticker):
    return SECTOR_MAP.get(ticker, 'Outros')

edges_df['setor_1'] = edges_df['ativo_1'].apply(get_sector)
edges_df['setor_2'] = edges_df['ativo_2'].apply(get_sector)
edges_df['mesmo_setor'] = edges_df['setor_1'] == edges_df['setor_2']

# Filtrar apenas para ativos com setor mapeado
known = edges_df[(edges_df['setor_1'] != 'Outros') & (edges_df['setor_2'] != 'Outros')]

if len(known) > 10:
    intra = known[known['mesmo_setor']]['distancia'].mean()
    inter = known[~known['mesmo_setor']]['distancia'].mean()

    print(f'Distância média intra-setor : {intra:.4f}')
    print(f'Distância média inter-setor : {inter:.4f}')
    print(f'Ratio inter/intra           : {inter/intra:.2f}x')

    sanity_pass = intra < inter
    status = '✅' if sanity_pass else '❌'
    print(f'{status} Distância intra-setor < inter-setor: {sanity_pass}')

    if not sanity_pass:
        print('ATENÇÃO: A estrutura setorial não aparece no grafo!')
        print('Possíveis causas: janela de correlação muito curta, dados sujos.')
else:
    print('Poucos pares com setor mapeado para análise intra/inter-setor.')

In [ ]:
# ─── 6. BENCHMARK DE TEMPO ───────────────────────────────────
# Testar tempo para diferentes tamanhos de universo
print('Benchmark de tempo de build:')

for n_assets in [30, 60, 100, 150]:
    sample_tickers = universe[:n_assets]
    t0 = time.perf_counter()
    G_bench = builder.build(window, eligible_tickers=sample_tickers)
    t = time.perf_counter() - t0
    status = '✅' if t < 5 else '⚠️'
    print(f'  {status} {n_assets:3d} ativos: {t:.2f}s')

print('\nSe > 5s para 150+ ativos: considerar Sinkhorn no Sprint 3.')

In [ ]:
# ─── 7. CHECKLIST FINAL ──────────────────────────────────────
print('\n' + '='*60)
print('CHECKLIST DE SAÍDA DO SPRINT 2')
print('='*60)

G_test = builder.build(window, eligible_tickers=universe)
sanity_checks = builder.validate_graph_sanity(G_test)

# Benchmark time
t0 = time.perf_counter()
builder.build(window, eligible_tickers=universe[:80])
build_time = time.perf_counter() - t0

checks = {
    'Grafo MST é conexo':                    sanity_checks.get('grafo_conexo', False),
    'MST tem n-1 arestas':                   sanity_checks.get('mst_n_menos_1', False),
    'Distribuições de Markov válidas':       sanity_checks.get('markov_validas', False),
    f'Tempo de build < 5s ({build_time:.1f}s)': build_time < 5.0,
}

# Adicionar checks de pares de vizinhos
for key, val in sanity_checks.items():
    if key not in checks and val is not None:
        checks[key] = val

all_pass = True
for check, result in checks.items():
    status = '✅' if result else '❌'
    if not result:
        all_pass = False
    print(f'{status} {check}')

print('\n' + '─'*60)
if all_pass:
    print('✅ SPRINT 2 CONCLUÍDO — Autorizado a abrir src/curvature/')
else:
    print('❌ SPRINT 2 INCOMPLETO — Corrija antes de calcular curvatura')
print('─'*60)